[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/cdg-ime/blob/main/Mineração.ipynb)

# Ciência de Dados Geoespaciais - Mineração

**Maj Diego - 2° Semestre / 2026**

**Objetivos**

1. Tratar dados para análise exploratória;
2. Selecionar modelos mais adequados à solução.

**Referência:**
- HAN, J.; KAMBER, M.; PEI, J. *Data Mining: Concepts and Techniques*. Morgan Kaufmann, 2011.
- REY, S.J.; ARRIBAS-BEL, D.; WOLF, L.J. *Geographic Data Science with Python*. CRC Press, 2023.
- ANSELIN, L. *Spatial Econometrics: Methods and Models*. Springer, 1988.

## O Contexto

Com os dados **prospectados** e **pré-processados**, chegamos à etapa de **mineração** — extrair conhecimento e padrões a partir dos dados.

```
┌─────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│  Prospecção │───▶│ Pré-Processo │───▶│  Mineração   │───▶│  Modelagem   │───▶│  Comunicação │
└─────────────┘    └──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
                                           ← você está aqui
```

A mineração tem dois grandes blocos:

| Bloco | Objetivo | Saída |
|-------|----------|-------|
| **Análise Exploratória (EDA)** | Entender distribuições, correlações e padrões espaciais | Gráficos, estatísticas, mapas |
| **Seleção de Modelo** | Escolher o algoritmo mais adequado ao problema | Modelo treinado e avaliado |

### Dataset desta aula

Usaremos um dataset sintético de **municípios do estado do Rio de Janeiro** com atributos socioeconômicos e geoespaciais, representativo de problemas reais de análise territorial.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (mean_absolute_error, r2_score,
                              classification_report, confusion_matrix)

np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Dataset sintético: 20 municípios do RJ ───────────────────
n = 20
municipios = [
    'Rio de Janeiro','Niterói','Duque de Caxias','Nova Iguaçu','Belford Roxo',
    'São Gonçalo','Campos dos Goytacazes','Petrópolis','Volta Redonda','Macaé',
    'Angra dos Reis','Cabo Frio','Resende','Nova Friburgo','Teresópolis',
    'Itaboraí','Maricá','Barra Mansa','Queimados','Magé'
]
lat = [-22.90,-22.88,-22.78,-22.75,-22.76,-22.83,-21.75,-22.51,-22.52,-22.37,
       -23.01,-22.88,-22.47,-22.28,-22.41,-22.71,-22.92,-22.54,-22.71,-22.65]
lon = [-43.17,-43.10,-43.31,-43.45,-43.40,-43.05,-41.33,-43.18,-44.08,-41.79,
       -44.32,-42.02,-44.45,-42.53,-42.97,-42.86,-42.82,-44.17,-43.55,-43.03]

df = pd.DataFrame({
    'municipio'      : municipios,
    'lat'            : lat,
    'lon'            : lon,
    'populacao'      : np.random.randint(50_000, 6_800_000, n),
    'pib_per_capita' : np.round(np.random.uniform(15_000, 85_000, n), 2),
    'idhm'           : np.round(np.random.uniform(0.63, 0.87, n), 3),
    'area_km2'       : np.round(np.random.uniform(80, 4500, n), 1),
    'cobertura_veg_pct': np.round(np.random.uniform(5, 75, n), 1),
    'precip_anual_mm': np.round(np.random.uniform(900, 2200, n), 1),
    'altitude_media_m': np.random.randint(5, 900, n),
    'dist_capital_km': np.round(
        np.sqrt((np.array(lat) - (-22.90))**2 + (np.array(lon) - (-43.17))**2) * 111, 1
    )
})

# Variável-alvo: crescimento populacional (correlacionado com IDH e PIB)
df['cresc_pop_pct'] = np.round(
    0.5 * df['idhm'] * 20 + 0.3 * (df['pib_per_capita'] / 10_000)
    - 0.1 * df['dist_capital_km'] + np.random.normal(0, 1.5, n), 2
)

# Classe de desenvolvimento (target para classificação)
df['classe_dev'] = pd.cut(
    df['idhm'],
    bins=[0, 0.699, 0.799, 1.0],
    labels=['Médio', 'Alto', 'Muito Alto']
)

gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(lo, la) for lo, la in zip(df['lon'], df['lat'])],
    crs='EPSG:4326'
)

print(f"Shape: {df.shape}")
df.head()

## 1. Tratar dados para análise exploratória

A **Análise Exploratória de Dados (EDA)** precede qualquer modelagem. Seu objetivo é *entender* os dados antes de *explicá-los*. Em ciência de dados geoespaciais, a EDA tem uma dimensão extra: além das estatísticas clássicas, investigamos **onde** os padrões ocorrem.

### 1.1 EDA Univariada — distribuição de cada variável

In [ ]:
# ── Estatísticas descritivas ──────────────────────────────────
print(df.describe().T.round(2))

In [ ]:
# ── Histogramas das variáveis numéricas ───────────────────────
num_cols = ['populacao','pib_per_capita','idhm','cobertura_veg_pct',
            'precip_anual_mm','altitude_media_m','dist_capital_km','cresc_pop_pct']

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.flat, num_cols):
    ax.hist(df[col], bins=8, edgecolor='white', color='steelblue')
    ax.set_title(col, fontsize=9)
    ax.axvline(df[col].mean(),   color='red',    linestyle='--', linewidth=1, label='média')
    ax.axvline(df[col].median(), color='orange', linestyle=':',  linewidth=1, label='mediana')

axes[0,0].legend(fontsize=7)
plt.suptitle('Distribuição das variáveis numéricas', fontsize=12)
plt.tight_layout()
plt.show()

### 1.2 EDA Bivariada — relações entre variáveis

O **coeficiente de correlação de Pearson** ($r$) mede a força e a direção da relação linear entre duas variáveis contínuas:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i-\bar{x})^2 \cdot \sum (y_i-\bar{y})^2}} \quad \in [-1, 1]$$

> ⚠️ Correlação **não implica causalidade**. Sempre interprete no contexto do domínio.

In [ ]:
# ── Mapa de calor de correlações ──────────────────────────────
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Apenas triângulo inferior
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Matriz de correlação de Pearson', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter plots das variáveis mais correlacionadas com o alvo
pares = [('idhm', 'cresc_pop_pct'), ('pib_per_capita', 'cresc_pop_pct'),
         ('dist_capital_km', 'cresc_pop_pct'), ('altitude_media_m', 'precip_anual_mm')]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (x, y) in zip(axes, pares):
    ax.scatter(df[x], df[y], alpha=0.7, edgecolors='white', s=60, color='steelblue')
    # Linha de tendência
    m, b = np.polyfit(df[x], df[y], 1)
    xs = np.linspace(df[x].min(), df[x].max(), 100)
    ax.plot(xs, m*xs + b, 'r--', linewidth=1.5)
    r = df[[x,y]].corr().iloc[0,1]
    ax.set_title(f'r = {r:.2f}', fontsize=10)
    ax.set_xlabel(x, fontsize=8)
    ax.set_ylabel(y, fontsize=8)

plt.suptitle('Diagramas de dispersão com linha de tendência', fontsize=11)
plt.tight_layout()
plt.show()

### 1.3 EDA Espacial — onde os padrões ocorrem?

Em dados geoespaciais, a EDA deve incluir **visualização cartográfica** e a investigação de **autocorrelação espacial** — a tendência de valores semelhantes se agruparem geograficamente.

#### Mapas coropléticos

Codificam uma variável numérica em cor, revelando padrões territoriais invisíveis em tabelas.

In [ ]:
# ── Mapas coropléticos: 4 variáveis ──────────────────────────
variaveis = [
    ('idhm',             'IDHM',                   'RdYlGn'),
    ('pib_per_capita',   'PIB per capita (R$)',     'YlOrBr'),
    ('precip_anual_mm',  'Precipitação anual (mm)', 'Blues'),
    ('cobertura_veg_pct','Cobertura vegetal (%)',   'Greens'),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, (col, titulo, cmap) in zip(axes, variaveis):
    gdf.plot(column=col, ax=ax, legend=True, cmap=cmap,
             markersize=80, edgecolor='black', linewidth=0.5,
             legend_kwds={'shrink': 0.7, 'label': titulo})
    for _, row in gdf.iterrows():
        ax.annotate(row['municipio'][:5], xy=(row.geometry.x, row.geometry.y),
                    xytext=(2, 4), textcoords='offset points', fontsize=5.5)
    ax.set_title(titulo, fontsize=9)
    ax.set_axis_off()

plt.suptitle('Análise Espacial Exploratória — Municípios do RJ', fontsize=12)
plt.tight_layout()
plt.show()

#### Autocorrelação Espacial — Índice de Moran

O **Índice de Moran ($I$)** mede se valores semelhantes tendem a se agrupar no espaço:

$$I = \frac{n}{\sum_i \sum_j w_{ij}} \cdot \frac{\sum_i \sum_j w_{ij}(x_i - \bar{x})(x_j - \bar{x})}{\sum_i (x_i - \bar{x})^2}$$

| Valor de $I$ | Interpretação |
|-------------|---------------|
| $I > 0$ | Autocorrelação positiva (valores similares agrupados) |
| $I \approx 0$ | Distribuição aleatória |
| $I < 0$ | Autocorrelação negativa (valores distintos vizinhos) |

In [ ]:
# ── Índice de Moran Global (implementação manual) ─────────────
from scipy.spatial.distance import cdist

def moran_global(valores, coords, k=4):
    """
    Calcula o índice de Moran usando matriz de pesos KNN (k vizinhos mais próximos).
    coords: array (n, 2) com [lat, lon]
    """
    n   = len(valores)
    x   = np.array(valores, dtype=float)
    xm  = x - x.mean()

    # Distâncias entre todos os pares
    D = cdist(coords, coords)
    np.fill_diagonal(D, np.inf)

    # Matriz de pesos binária KNN
    W = np.zeros((n, n))
    for i in range(n):
        vizinhos = np.argsort(D[i])[:k]
        W[i, vizinhos] = 1
    W = (W + W.T)  # Simetrizar
    W = W / W.sum()  # Normalizar

    num = n * np.sum(W * np.outer(xm, xm))
    den = W.sum() * np.sum(xm**2)
    return num / den

coords = df[['lat','lon']].values
for col in ['idhm', 'pib_per_capita', 'precip_anual_mm']:
    I = moran_global(df[col].values, coords)
    interpretacao = 'agrupamento espacial' if I > 0.1 else ('dispersão' if I < -0.1 else 'aleatório')
    print(f"Moran's I ({col:25s}): {I:+.3f}  →  {interpretacao}")

### 1.4 Engenharia de Features Geoespaciais

Frequentemente, as features mais informativas **não estão no dado original**, mas são derivadas da geometria ou da relação espacial entre feições.

| Feature derivada | Como calcular | Exemplo de uso |
|-----------------|---------------|----------------|
| Distância a uma feição | `geometry.distance()` | Distância à capital, ao mar, a rodovias |
| Densidade de vizinhança | Contagem em buffer | Nº de equipamentos no raio de 5 km |
| Lag espacial | Média dos vizinhos | IDH médio dos municípios vizinhos |
| Área e perímetro | `geometry.area`, `.length` | Compacidade territorial |
| Centroide | `geometry.centroid` | Coordenadas do centro geométrico |

In [ ]:
# ── Feature: lag espacial do IDHM (média dos 3 vizinhos mais próximos) ──
from scipy.spatial import cKDTree

tree = cKDTree(coords)
k = 3
dists, idxs = tree.query(coords, k=k+1)  # k+1 porque o próprio ponto é retornado
vizinhos_idx = idxs[:, 1:]               # Exclui o próprio ponto

df['idhm_lag'] = df['idhm'].values[vizinhos_idx].mean(axis=1).round(3)
df['pib_lag']  = df['pib_per_capita'].values[vizinhos_idx].mean(axis=1).round(0)

# ── Densidade relativa: população / área ──────────────────────
df['dens_demografica'] = (df['populacao'] / df['area_km2']).round(1)

print("Features derivadas:")
print(df[['municipio','idhm','idhm_lag','pib_per_capita','pib_lag','dens_demografica']].to_string(index=False))

## 2. Selecionar modelos mais adequados à solução

### 2.1 Mapa mental: qual algoritmo usar?

A escolha do modelo depende fundamentalmente da **natureza do problema** e da **variável-alvo**:

```
                        ┌─── Tenho rótulos? ───┐
                       Sim                     Não
                        │                       │
             ┌─── Aprendizado ───┐      Aprendizado Não
             │    Supervisionado │       Supervisionado
             │                  │             │
        Alvo contínuo    Alvo categórico  Agrupamento
             │                  │             │
        Regressão        Classificação   Clusterização
      (Linear, RF,       (RF, SVM,       (K-Means,
       XGBoost...)        KNN...)         DBSCAN...)
```

### 2.2 Considerações espaciais na seleção de modelos

Dados geoespaciais violam frequentemente a **premissa de independência** dos modelos clássicos. Dois problemas principais:

| Problema | Descrição | Solução |
|----------|-----------|--------|
| **Autocorrelação espacial** | Resíduos correlacionados no espaço | Modelos espaciais, features de lag |
| **MAUP** (Problema da Unidade de Área Modificável) | Resultado muda com a escala/agregação | Testar múltiplas escalas |

> **Regra prática:** após qualquer modelo, plote os **resíduos no mapa**. Se houver padrão espacial visível, o modelo ainda não capturou toda a estrutura dos dados.

---

### 2.3 Clusterização espacial — K-Means e DBSCAN

**Problema:** agrupar municípios com perfis socioeconômicos e espaciais similares (aprendizado **não supervisionado**).

#### K-Means
Minimiza a variância intra-cluster. Requer que o número de clusters $k$ seja definido a priori. Usa o **Método do Cotovelo** para escolher $k$.

In [ ]:
# ── Features para clusterização ───────────────────────────────
feat_cluster = ['idhm','pib_per_capita','dens_demografica',
                'cobertura_veg_pct','precip_anual_mm','dist_capital_km']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feat_cluster])

# ── Método do Cotovelo ────────────────────────────────────────
inercias = []
Ks = range(2, 8)
for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cotovelo
axes[0].plot(Ks, inercias, 'o-', color='steelblue')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inércia')
axes[0].set_title('Método do Cotovelo')
axes[0].axvline(3, color='red', linestyle='--', label='k escolhido = 3')
axes[0].legend()

# Mapa dos clusters
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_kmeans'] = km_final.fit_predict(X_scaled)
gdf['cluster_kmeans'] = df['cluster_kmeans']

cores_cluster = {0: '#e74c3c', 1: '#2980b9', 2: '#27ae60'}
for c, cor in cores_cluster.items():
    subset = gdf[gdf['cluster_kmeans'] == c]
    subset.plot(ax=axes[1], color=cor, markersize=80,
                edgecolor='black', label=f'Cluster {c}')
    for _, row in subset.iterrows():
        axes[1].annotate(row['municipio'][:6],
                         xy=(row.geometry.x, row.geometry.y),
                         xytext=(2, 4), textcoords='offset points', fontsize=6)
axes[1].legend()
axes[1].set_title('Clusters K-Means (k=3) no espaço geográfico')
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

# Perfil de cada cluster
print("\nPerfil médio por cluster:")
print(df.groupby('cluster_kmeans')[feat_cluster].mean().round(2))

#### DBSCAN — Clusterização baseada em densidade

Diferente do K-Means, o **DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*):
- **Não requer** $k$ definido a priori
- Detecta clusters de **forma arbitrária**
- Identifica **ruído/outliers** (rótulo `-1`)
- Especialmente adequado para **dados espaciais**

Parâmetros principais:
- `eps`: raio da vizinhança
- `min_samples`: mínimo de pontos para formar um núcleo

In [ ]:
# ── DBSCAN sobre coordenadas geográficas (em graus) ───────────
# eps=1.5 graus ≈ ~165 km; min_samples=2
X_geo = df[['lat','lon']].values

db = DBSCAN(eps=1.5, min_samples=2)
df['cluster_dbscan'] = db.fit_predict(X_geo)
gdf['cluster_dbscan'] = df['cluster_dbscan']

n_clusters = len(set(df['cluster_dbscan'])) - (1 if -1 in df['cluster_dbscan'].values else 0)
n_ruido    = (df['cluster_dbscan'] == -1).sum()
print(f"Clusters encontrados: {n_clusters}")
print(f"Pontos de ruído (isolados): {n_ruido}")
print(df[['municipio','cluster_dbscan']].to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 5))
rotulos_unicos = sorted(df['cluster_dbscan'].unique())
palette = cm.tab10.colors
for i, rotulo in enumerate(rotulos_unicos):
    cor   = 'gray' if rotulo == -1 else palette[i % len(palette)]
    label = 'Ruído' if rotulo == -1 else f'Cluster {rotulo}'
    gdf[gdf['cluster_dbscan'] == rotulo].plot(
        ax=ax, color=cor, markersize=70, edgecolor='black', label=label
    )
ax.legend()
ax.set_title('DBSCAN — Clusterização por proximidade geográfica')
ax.set_axis_off()
plt.tight_layout()
plt.show()

### 2.4 Regressão — prever crescimento populacional

**Problema:** prever `cresc_pop_pct` a partir de atributos socioeconômicos e espaciais (aprendizado **supervisionado, alvo contínuo**).

Comparamos dois modelos:
- **Regressão Linear** — simples, interpretável, assume relação linear
- **Random Forest Regressor** — não-linear, robusto a outliers, captura interações

In [ ]:
feat_reg = ['idhm','pib_per_capita','dist_capital_km',
            'altitude_media_m','idhm_lag','pib_lag','dens_demografica']

X = df[feat_reg].values
y = df['cresc_pop_pct'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

modelos = {
    'Regressão Linear'  : LinearRegression(),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42)
}

print(f"{'Modelo':<22} {'MAE':>8} {'R²':>8}  (conjunto de teste)")
print('-' * 44)
resultados = {}
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    print(f"{nome:<22} {mae:>8.3f} {r2:>8.3f}")
    resultados[nome] = {'modelo': modelo, 'y_pred': y_pred, 'mae': mae, 'r2': r2}

# ── Importância de features (Random Forest) ───────────────────
rf = resultados['Random Forest']['modelo']
importancias = pd.Series(rf.feature_importances_, index=feat_reg).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Valores reais vs preditos
for ax, (nome, res) in zip(axes, resultados.items()):
    ax.scatter(y_test, res['y_pred'], alpha=0.7, edgecolors='white', s=60, color='steelblue')
    lims = [min(y_test.min(), res['y_pred'].min()), max(y_test.max(), res['y_pred'].max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5)
    ax.set_title(f"{nome}\nMAE={res['mae']:.3f}  R²={res['r2']:.3f}", fontsize=9)
    ax.set_xlabel('Valor real')
    ax.set_ylabel('Predito')

plt.suptitle('Regressão: real vs predito', fontsize=11)
plt.tight_layout()
plt.show()

# Importância
fig, ax = plt.subplots(figsize=(6, 4))
importancias.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Importância das features (Random Forest)')
ax.set_xlabel('Importância relativa')
plt.tight_layout()
plt.show()

### 2.5 Classificação — identificar classe de desenvolvimento

**Problema:** prever `classe_dev` (Médio / Alto / Muito Alto) com base nos atributos (aprendizado **supervisionado, alvo categórico**).

A **validação cruzada** (*k-fold cross-validation*) é fundamental quando o dataset é pequeno — usa todo o dado para treino e teste de forma rotativa, fornecendo estimativas mais robustas do desempenho real.

In [ ]:
from sklearn.preprocessing import LabelEncoder

feat_clf = ['idhm','pib_per_capita','dist_capital_km','dens_demografica','idhm_lag']

le = LabelEncoder()
y_clf = le.fit_transform(df['classe_dev'].astype(str))
X_clf = df[feat_clf].values

clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Validação cruzada (5-fold)
scores = cross_val_score(clf, X_clf, y_clf, cv=5, scoring='accuracy')
print(f"Acurácia por fold: {scores.round(2)}")
print(f"Acurácia média: {scores.mean():.2f} ± {scores.std():.2f}")

# Treino final e matriz de confusão
X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)
clf.fit(X_tr, y_tr)
y_pred_clf = clf.predict(X_te)

print("\nRelatório de classificação:")
print(classification_report(y_te, y_pred_clf, target_names=le.classes_))

# Matriz de confusão
cm_arr = confusion_matrix(y_te, y_pred_clf)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predito')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão — Random Forest')
plt.tight_layout()
plt.show()

### 2.6 Diagnóstico espacial dos resíduos

Após qualquer modelo, é boa prática mapear os resíduos. Um **padrão espacial nos resíduos** indica que o modelo não capturou toda a estrutura dos dados — e que variáveis espaciais adicionais podem melhorar o desempenho.

In [ ]:
# ── Resíduos da Regressão Linear sobre todos os dados ────────
modelo_lin = modelos['Regressão Linear']
modelo_lin.fit(X, y)  # Re-treina com todos os dados para plotar
residuos = y - modelo_lin.predict(X)

gdf['residuo'] = residuos

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mapa dos resíduos
gdf.plot(column='residuo', ax=axes[0], legend=True,
         cmap='RdBu', markersize=100, edgecolor='black',
         legend_kwds={'label': 'Resíduo', 'shrink': 0.7})
for _, row in gdf.iterrows():
    axes[0].annotate(row['municipio'][:5],
                     xy=(row.geometry.x, row.geometry.y),
                     xytext=(2, 4), textcoords='offset points', fontsize=6)
axes[0].set_title('Mapa de resíduos da Regressão Linear\n(azul = sub, vermelho = superestimado)')
axes[0].set_axis_off()

# Distribuição dos resíduos
axes[1].hist(residuos, bins=8, edgecolor='white', color='steelblue')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Distribuição dos resíduos')
axes[1].set_xlabel('Resíduo')
axes[1].set_ylabel('Frequência')

plt.suptitle('Diagnóstico Espacial dos Resíduos', fontsize=12)
plt.tight_layout()
plt.show()

# Moran dos resíduos
I_res = moran_global(residuos, coords)
print(f"Moran's I dos resíduos: {I_res:+.3f}")
print("→ Se |I| > 0.1, considere adicionar features de lag espacial ou usar um modelo espacial.")

## Complemento

### Guia de seleção de modelos para dados geoespaciais

| Tipo de problema | Algoritmos recomendados | Atenção espacial |
|-----------------|------------------------|------------------|
| Agrupamento por perfil | K-Means, Agglomerative | Verificar coesão espacial dos clusters |
| Agrupamento por proximidade | DBSCAN, HDBSCAN | Converter para CRS métrico antes |
| Regressão espacialmente estacionária | OLS, Random Forest | Mapear resíduos |
| Regressão com não-estacionariedade | GWR (Geographically Weighted Regression) | Coeficientes variam no espaço |
| Classificação territorial | Random Forest, XGBoost | Usar lag espacial como feature |
| Detecção de hotspots | LISA (Local Moran), Getis-Ord Gi* | Análise local, não global |

### Checklist antes de publicar um modelo

- [ ] Dados divididos corretamente (treino / teste)
- [ ] Padronização/normalização aplicada quando necessário
- [ ] Validação cruzada usada para datasets pequenos
- [ ] Resíduos mapeados e analisados
- [ ] Índice de Moran dos resíduos próximo de zero
- [ ] Feature importance analisada
- [ ] Métricas reportadas com desvio-padrão (cross-val)

### Bibliotecas especializadas

```python
# PySAL: autocorrelação espacial, LISA, Moran, GWR
from esda.moran import Moran
from mgwr.gwr import GWR

# pysal / libpysal: matrizes de pesos espaciais
from libpysal.weights import KNN, Queen
w = KNN.from_dataframe(gdf, k=4)

# HDBSCAN: clusterização hierárquica por densidade
import hdbscan
clusterer = hdbscan.HDBSCAN(min_cluster_size=3)
labels = clusterer.fit_predict(X_geo)

# folium / plotly: mapas interativos para exploração
import folium
m = folium.Map(location=[-22.5, -43.0], zoom_start=8)
```

## Lista de exercícios complementares

**Exercício 1.** Com o dataset desta aula, calcule a correlação de Pearson entre todas as variáveis numéricas e identifique o par com maior e menor correlação. Interprete cada resultado no contexto territorial.

**Exercício 2.** Aplique K-Means com $k \in \{2, 3, 4, 5\}$ usando apenas as coordenadas geográficas (`lat`, `lon`). Compare os mapas resultantes com os clusters obtidos usando variáveis socioeconômicas. Os agrupamentos coincidem? O que isso indica?

**Exercício 3.** Aplique DBSCAN variando `eps` entre 0.5 e 3.0 (passo 0.5). Para cada configuração, registre o número de clusters e de pontos de ruído. Qual configuração você considera mais adequada para o RJ? Justifique.

**Exercício 4.** Treine um `RandomForestRegressor` para prever `precip_anual_mm` usando apenas variáveis topográficas e de localização (`altitude_media_m`, `lat`, `lon`, `dist_capital_km`). Calcule o MAE e R² por validação cruzada (5-fold). Mapeie os resíduos e calcule o Índice de Moran dos resíduos.

**Exercício 5.** Crie manualmente uma feature de *lag espacial* para a variável `pib_per_capita` usando os 5 vizinhos mais próximos. Adicione essa feature ao modelo de regressão e compare o R² antes e depois. A autocorrelação espacial dos resíduos diminuiu?

**Exercício 6 (desafio).** Implemente a **GWR simplificada** (Geographically Weighted Regression): para cada município $i$, treine uma regressão linear ponderada pelos inversos das distâncias aos demais municípios. O alvo é `cresc_pop_pct`. Compare os coeficientes obtidos para `idhm` em municípios da região serrana versus litoral. Os coeficientes são estacionários?